# Colab runner

This notebook only mounts Drive, pulls the repository, installs the package, and starts scripts.
All logic lives in the package `ddd` and in `scripts/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # needs a click by the owner

In [ ]:
REPO_URL = 'https://github.com/SauHin/driver-distraction.git'
REPO = '/content/driver-distraction'
DRIVE = '/content/drive/MyDrive/driver-distraction'  # data, manifests, logs, runs
%env REPO={REPO}
%env DRIVE={DRIVE}

In [ ]:
%%bash
if [ -d "$REPO/.git" ]; then git -C "$REPO" pull --ff-only; else git clone "$REPO_URL" "$REPO"; fi
git -C "$REPO" log -1 --oneline

In [ ]:
%%bash
pip install -q -e "$REPO[test,train]"
cd "$REPO" && python -m pytest -q

In [ ]:
%%bash
# Copy the Kaggle zip from Drive to the local disk, unzip it, then compare with the manifest of the laptop.
mkdir -p "$REPO/data/raw/statefarm"
cp "$DRIVE/data/raw/statefarm/state-farm-distracted-driver-detection.zip" "$REPO/data/raw/statefarm/"
cd "$REPO/data/raw/statefarm" && unzip -q -o state-farm-distracted-driver-detection.zip
cd "$REPO" && python scripts/make_manifest.py data/raw/statefarm "$DRIVE/manifests/statefarm.csv" --check

In [ ]:
%%bash
# ImageNet weights of DeiT-III Base, the link of timm 0.6.12. See results/vitdd_public_files.md.
W="$REPO/data/external/weights/deit3_base_224_1k"
mkdir -p "$W" && cd "$W"
[ -f deit_3_base_224_1k.pth ] || wget -q https://dl.fbaipublicfiles.com/deit/deit_3_base_224_1k.pth
echo "92d4e4ac874725a51c703dce67379d90df4af3805a8b27ec44546e580152003c  deit_3_base_224_1k.pth" | sha256sum -c

In [ ]:
%%bash
# C6: one epoch of the R1 config on the development drivers. No test driver is read.
# The GPU log samples nvidia-smi every 30 seconds.
mkdir -p "$DRIVE/logs" "$DRIVE/runs"
nohup nvidia-smi --query-gpu=timestamp,name,utilization.gpu,memory.used,memory.total --format=csv -l 30   > "$DRIVE/logs/c6_gpu.csv" 2>&1 &
cd "$REPO" && nohup python scripts/train.py configs/r1_vit.json --runs "$DRIVE/runs" --name c6_colab --epochs 1   > "$DRIVE/logs/c6_colab.log" 2>&1 &
echo "started, log: $DRIVE/logs/c6_colab.log"

In [ ]:
!tail -n 5 "$DRIVE/logs/c6_colab.log"; tail -n 3 "$DRIVE/logs/c6_gpu.csv"

In [ ]:
%%bash
# After the epoch ends: stop the GPU log and save the speed file on Drive.
pkill -f "nvidia-smi --query-gpu" || true
mkdir -p "$DRIVE/results/compute"
cd "$REPO" && python scripts/record_speed.py "$DRIVE/runs/c6_colab" --key colab   --note "R1 config, one full epoch. GPU use: $DRIVE/logs/c6_gpu.csv." --out "$DRIVE/results/compute/first_run_colab.json"